# CAMUS Baseline — Attention Residual U-Net

Thin notebook: all logic lives in the `iteris/` package. This file just installs deps,
loads config, and calls high-level functions.

**Kaggle setup**
1. Add CAMUS dataset to inputs (path used below: `/kaggle/input/datasets/anfaalhossain/camus/CAMUS`)
2. Add `iteris-pkg` Kaggle Dataset to inputs (upload the `iteris/` folder + `configs/` + `requirements.txt`)
3. Run the first cell, **restart the kernel**, then **Run All** from cell 2 onward

**To switch datasets:** change the YAML path in Cell 2. Nothing else changes.

## 0 · Setup
Lightweight install — uses Kaggle's default numpy/torch/scipy versions (no conflicts).
Run this cell once, then **Run All** (no kernel restart required).

In [ ]:
import subprocess
from pathlib import Path

# Find requirements.txt under iteris-pkg (handles any Kaggle wrapper layout)
req_files = list(Path('/kaggle/input').rglob('requirements.txt'))
req_file = next(
    (r for r in req_files if (r.parent / 'iteris' / '__init__.py').exists()),
    None,
)
if req_file is None:
    raise RuntimeError('iteris-pkg dataset not attached. Sidebar → Data → + Add Input.')

# Minimal install — only the packages our code actually needs. Don't touch numpy/torch/scipy:
# Kaggle's defaults are current and downgrading them creates pip resolver conflicts.
# Our code uses pure-torch metrics (no scipy dependency), so no version pinning required.
subprocess.run(
    ['pip', 'install', '-r', str(req_file), '--quiet', '--upgrade'],
    check=True,
)
print('✓ Setup complete. No kernel restart needed — Run All from cell 1 onwards.')

## 1 · Setup — package on path + config

In [ ]:
import sys
from pathlib import Path

# Find iteris/__init__.py anywhere under /kaggle/input — robust to any wrapper depth or name
init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError(
        'iteris package not found anywhere under /kaggle/input. '
        'Is the iteris-pkg dataset attached? Sidebar → Data → + Add Input.'
    )
PKG_ROOT = init_files[0].parent.parent
sys.path.insert(0, str(PKG_ROOT))
print(f'Package root : {PKG_ROOT}')

from iteris.config import load_config
from iteris.utils  import get_device

cfg_files = list(PKG_ROOT.rglob('configs/camus.yaml'))
cfg = load_config(str(cfg_files[0]))

# Override Kaggle-specific paths (these change per session / dataset upload)
cfg['data_root']      = '/kaggle/input/datasets/anfaalhossain/camus/CAMUS'
cfg['checkpoint_dir'] = '/kaggle/working'

print(f'Dataset      : {cfg["dataset"]}')
print(f'Image size   : {cfg["image_size"]}')
print(f'Classes      : {cfg["num_classes"]} — {cfg["class_names"]}')
print(f'Epochs       : {cfg["epochs"]}  batch {cfg["batch_size"]}  lr {cfg["lr"]}')
print(f'Device       : {get_device()}')

## 2 · Train
End-to-end: ingestion → patient-level split → MONAI transforms → cached dataloaders →
Attention Residual U-Net training with cosine schedule + early stopping → best checkpoint saved.

In [ ]:
from iteris.training import run_training

result = run_training(cfg)
model         = result['model']
history       = result['history']
best_dice     = result['best_dice']
test_loader   = result['test_loader']
checkpoint    = result['checkpoint']

print(f'\n✓ Training done. Best val Dice = {best_dice:.4f}  |  Checkpoint: {checkpoint}')

## 3 · Learning curves

In [ ]:
from iteris.visualization import plot_learning_curves
plot_learning_curves(history, cfg, target_dice=0.85)

## 4 · Test-set evaluation
Per-patient Dice + HD95 (pure-torch implementation, no scipy). Writes the CSV that
Week 4 statistical tests and DRL agents consume.

In [ ]:
from iteris.evaluation import evaluate_test_set
scores_df = evaluate_test_set(model, test_loader, cfg)
print(scores_df.head())

## 5 · Export predicted masks
One `.npy` per test sample. These become the DRL environment's `init_mask`.

In [ ]:
from iteris.evaluation import export_predicted_masks
export_predicted_masks(model, test_loader, cfg)

## 6 · Qualitative grid

In [ ]:
from iteris.visualization import plot_qualitative_grid
plot_qualitative_grid(model, test_loader, cfg, n_show=4)

## 7 · Summary JSON
Snapshot consumed by downstream notebooks.

In [ ]:
from iteris.evaluation import save_summary_json
save_summary_json(history, scores_df, cfg, best_dice)